In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_members_ssp(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "MemberId String"
        # ",MemberKey String"
        # ",MemberCode Int"
        # ",SourceSystem String"
        # ",IngestionTimestamp Timestamp"
        # ",BatchId String"
        # ",MemberEmailAddress String"
        # ",IsEnabledMember Boolean"
        # ",MemberCreatedDate Timestamp"
        # ",MemberGuid String"
        # ",MemberName String"
        # ",MemberPassword String"
        # ",IsExternalMemberFlag Boolean"
        # ",MemberSecurityStamp String"
        # ",RegistrationStatus String"
        # ",MemberPhoneNumber String"
        # ",PartyType String"
        # ",AzureB2CGUID String"
        # ",LastModifiedDate Timestamp"
        # ",MockedTaxRegistrationNumber String"
        # ",LastLoginDate Timestamp"
        # ",IsCorporateFlag Boolean"
        # ",UserGuid String"
        # ",BirthDate Date"
        # ",BirthYear String"
        # ",BirthPlace String"
        # ",FatherName String"
        # ",FatherNameENG String"
        # ",MotherName String"
        # ",MotherNameENG String"
        # ",FirstName String"
        # ",FirstNameENG String"
        # ",LastName String"
        # ",LastNameENG String"
        # ",Occupation String"
        # ",Address String"
        # ",ContactAddress String"
        # ",EmailAddress String"
        # ",MobilePhoneNumber String"
        # ",TaxRegistrationNumber String"
        # ",DocumentType String"
        # ",DocumentNumber String"
        # ",DocumentExpiredDate String"
        # ",DocumentIssuingAuthority String"
        # ",CreatedDate Timestamp"
        # ",ModifiedDate Timestamp"
        # ",CompanyType String"
        # ",PhoneNumber String"
        # ",IsLegalRep Boolean"
        # ",CompanyName String"
        # ",CompanyTaxRegistrationNumber String"
        # ",CompanyLandLine String"
        # ",CompanyHeadOfficeAddress String"
        # ",RequestKey String"
        # ",ApplicationKey String"
        # ",WorkflowApplicationKey String"
        # ",WorkflowApplicationStatus String"
        # ",WorkflowApplicationTitle String"
        # ",WorkflowApplicationName String"
        # ",BusinessDate Date"
        #  )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Members"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "ssp",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== cmsmember ==================

        cmsmember_extra_filters = None

        if cmsmember_extra_filters is not None:
            cmsmember_extra_filters.append(date_filter)
            cmsmember_final_filters = " AND ".join(cmsmember_extra_filters)
        else:
            cmsmember_final_filters = date_filter

        cmsmember_selected_fields = [
            "MemberID",
            "MemberEmail",
            "MemberEnabled",
            "MemberCreated",
            "MemberGuid",
            "MemberName",
            "MemberPassword",
            "MemberIsExternal",
            "MemberSecurityStamp",
            "RegistrationStatus",
            "Phone",
            "DebtorType",
            "AzureB2CGUID",
            "LastModified",
            "MockedAFM",
            "LastLogin",
            "BusinessDate",
        ]

        cmsmember = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "ssp_CmsMember",
            cmsmember_final_filters,
            cmsmember_selected_fields,
        )

        # ================== kycdata ==================

        kycdata_extra_filters = None

        if kycdata_extra_filters is not None:
            kycdata_extra_filters.append(date_filter)
            kycdata_final_filters = " AND ".join(kycdata_extra_filters)
        else:
            kycdata_final_filters = date_filter

        kycdata_selected_fields = [
            "UserGuid",
            "BirthDate",
            "BirthYear",
            "BirthPlace",
            "FatherName",
            "FatherNameLatin",
            "MotherName",
            "MotherNameLatin",
            "Name",
            "NameLatin",
            "Surname",
            "SurnameLatin",
            "Profession",
            "Address",
            "ContactAddress",
            "Email",
            "Mobile",
            "Telephone",
            "VatNumber",
            "DocumentType",
            "DocumentNumber",
            "DocumentExpirationDate",
            "DocumentIssuePlace",
            "CreatedOn",
            "ModifiedOn",
            "BusinessDate",
        ]

        kycdata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "ssp_KYCData",
            kycdata_final_filters,
            kycdata_selected_fields,
        )

        # ================== CorporateData ==================

        corporatedata_extra_filters = None

        if corporatedata_extra_filters is not None:
            corporatedata_extra_filters.append(date_filter)
            corporatedata_final_filters = " AND ".join(corporatedata_extra_filters)
        else:
            corporatedata_final_filters = date_filter

        corporatedata_selected_fields = [
            "UserGuid",
            "FirstName",
            "LastName",
            "Email",
            "VatNumber",
            "CreatedOn",
            "ModifiedOn",
            "CompanyType",
            "Phone",
            "IsLegalRep",
            "CompanyName",
            "CompanyVatNumber",
            "CompanyLandLine",
            "CompanyHeadOfficeAddress",
            "RequestId",
            "ApplicationId",
            "WorkflowApplicationId",
            "WorkflowApplicationStatus",
            "WorkflowApplicationTitle",
            "WorkflowApplicationName",
            "BusinessDate",
        ]

        corporatedata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "ssp_CorporateData",
            corporatedata_final_filters,
            corporatedata_selected_fields,
        )

        # =============== Union & Joins =============

        kycdata = (
            kycdata.withColumnRenamed("Name", "FirstName")
            .withColumnRenamed("Surname", "LastName")
            .withColumnRenamed("Telephone", "Phone")
            .withColumn("CompanyType", lit(None))
            .withColumn("IsLegalRep", lit(None))
            .withColumn("CompanyName", lit(None))
            .withColumn("CompanyVatNumber", lit(None))
            .withColumn("CompanyLandLine", lit(None))
            .withColumn("CompanyHeadOfficeAddress", lit(None))
            .withColumn("RequestId", lit(None))
            .withColumn("ApplicationId", lit(None))
            .withColumn("WorkflowApplicationId", lit(None))
            .withColumn("WorkflowApplicationStatus", lit(None))
            .withColumn("WorkflowApplicationTitle", lit(None))
            .withColumn("WorkflowApplicationName", lit(None))
            .withColumn("IsCorporateFlag", lit(False))
        )

        corporatedata = (
            corporatedata.withColumn("BirthDate", lit(None))
            .withColumn("BirthYear", lit(None))
            .withColumn("BirthPlace", lit(None))
            .withColumn("FatherName", lit(None))
            .withColumn("FatherNameLatin", lit(None))
            .withColumn("MotherName", lit(None))
            .withColumn("MotherNameLatin", lit(None))
            .withColumn("NameLatin", lit(None))
            .withColumn("SurnameLatin", lit(None))
            .withColumn("Profession", lit(None))
            .withColumn("Address", lit(None))
            .withColumn("ContactAddress", lit(None))
            .withColumn("Mobile", lit(None))
            .withColumn("DocumentType", lit(None))
            .withColumn("DocumentNumber", lit(None))
            .withColumn("DocumentExpirationDate", lit(None))
            .withColumn("DocumentIssuePlace", lit(None))
            .withColumn("IsCorporateFlag", lit(True))
        )

        # Union
        kycdata_union_corporatedata = kycdata.unionByName(corporatedata)

        # Joins
        members_df = cmsmember.join(
            kycdata_union_corporatedata,
            cmsmember["MemberGuid"] == kycdata_union_corporatedata["UserGuid"],
            "LEFT",
        )

        # =============== Primary Keys ===============
        members_primary_key_fields = [
            members_df.MemberID,
            members_df.MemberGuid,
            lit("ssp"),
        ]

        members_key_fields = [
            members_df.MemberID,
            members_df.MemberGuid,
        ]

        members_df = members_df.withColumn(
            "MembersId",
            sha2(concat_ws(";", *members_primary_key_fields), 256),
        ).withColumn("MemberKey", concat_ws(";", *members_key_fields))

        # ================== Selection of Final Columns ==================
        members_df = members_df.select(
            members_df.MembersId.alias("MemberId"),
            members_df.MemberKey,
            members_df.MemberID.alias("MemberCode"),
            lit("ssp").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            members_df.MemberEmail.alias("MemberEmailAddress"),
            members_df.MemberEnabled.alias("IsEnabledMember"),
            members_df.MemberCreated.alias("MemberCreatedDate"),
            members_df.MemberGuid,
            members_df.MemberName,
            members_df.MemberPassword,
            members_df.MemberIsExternal.alias("IsExternalMemberFlag"),
            members_df.MemberSecurityStamp,
            members_df.RegistrationStatus,
            cmsmember.Phone.alias("MemberPhoneNumber"),
            members_df.DebtorType.alias("PartyType"),
            members_df.AzureB2CGUID,
            members_df.LastModified.alias("LastModifiedDate"),
            members_df.MockedAFM.alias("MockedTaxRegistrationNumber"),
            members_df.LastLogin.alias("LastLoginDate"),
            members_df.IsCorporateFlag,
            members_df.UserGuid,
            members_df.BirthDate,
            members_df.BirthYear,
            members_df.BirthPlace,
            members_df.FatherName,
            members_df.FatherNameLatin.alias("FatherNameENG"),
            members_df.MotherName,
            members_df.MotherNameLatin.alias("MotherNameENG"),
            members_df.FirstName,
            members_df.NameLatin.alias("FirstNameENG"),
            members_df.LastName,
            members_df.SurnameLatin.alias("LastNameENG"),
            members_df.Profession.alias("Occupation"),
            members_df.Address,
            members_df.ContactAddress,
            members_df.Email.alias("EmailAddress"),
            members_df.Mobile.alias("MobilePhoneNumber"),
            kycdata_union_corporatedata.Phone.alias("PhoneNumber"),
            members_df.VatNumber.alias("TaxRegistrationNumber"),
            members_df.DocumentType,
            members_df.DocumentNumber,
            members_df.DocumentExpirationDate.alias("DocumentExpiredDate"),
            members_df.DocumentIssuePlace.alias("DocumentIssuingAuthority"),
            members_df.CompanyType,
            members_df.IsLegalRep,
            members_df.CompanyName,
            members_df.CompanyVatNumber.alias("CompanyTaxRegistrationNumber"),
            members_df.CompanyLandLine,
            members_df.CompanyHeadOfficeAddress,
            members_df.RequestId.alias("RequestKey"),
            members_df.ApplicationId.alias("ApplicationKey"),
            members_df.WorkflowApplicationId.alias("WorkflowApplicationKey"),
            members_df.WorkflowApplicationStatus,
            members_df.WorkflowApplicationTitle,
            members_df.WorkflowApplicationName,
            members_df.CreatedOn.alias("CreatedDate"),
            members_df.ModifiedOn.alias("ModifiedDate"),
            cmsmember.BusinessDate,
        )

        # ================== Append ==================
        members_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_members_ssp(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)